# DVC debug — band-limited texture only

Strips the bone phantom out of the loop so the only inputs to `correlate()` are
the band-limited noise reference produced by `validate.synthetic.make_texture`
and a warped copy of it. This isolates the algorithm's behaviour from the
piecewise-constant scaffold of `default_phantom` and matches the operating
regime that the plan's 0.1-voxel accuracy budget is calibrated against.

Each section reports per-axis MAE and total RMSE/percentile errors. Variants:

1. Integer rigid shift (sub-voxel fit not exercised).
2. Fractional rigid shift `(0.5, -0.3, 0.7)` (clean Gaussian-fit budget).
3. The showcase shift `(1.5, 0.0, -2.0)` (matches the failing case).
4. Same as (3) plus the long-wavelength sinusoidal warp.

All runs use `window=32`, `overlap=0.5`, matching the showcase. The
boundary band is excluded from the error stats so the reflect-mode warp
near the volume edge does not pollute the budget.

In [ ]:
import numpy as np
from mamba_dvc.pipeline.correlate import correlate
from mamba_dvc.types import POIStatus, VoxelSpacing
from mamba_dvc.validate.synthetic import (
    compose,
    make_pair,
    make_texture,
    rigid_shift,
    sinusoidal,
)

shape = (96, 128, 128)
spacing = VoxelSpacing((1.0, 1.0, 1.0), "um")
window = 32
overlap = 0.5
boundary_band = window  # exclude POIs whose center sits within `window` of any face

In [ ]:
def interior_mask(positions: np.ndarray, shape: tuple[int, int, int], band: int) -> np.ndarray:
    """Boolean mask of POIs whose center is at least ``band`` voxels from any face."""
    z, y, x = positions[:, 0], positions[:, 1], positions[:, 2]
    return (
        (z >= band) & (z < shape[0] - band)
        & (y >= band) & (y < shape[1] - band)
        & (x >= band) & (x < shape[2] - band)
    )


def report(label: str, result, field_fn) -> None:
    truth = field_fn(result.positions)
    interior = interior_mask(result.positions, shape, boundary_band)
    valid = result.valid & interior
    n_total = result.positions.shape[0]
    n_valid_all = int(result.valid.sum())
    n_valid_interior = int(valid.sum())

    err = (result.displacements - truth)[valid]
    mag = np.linalg.norm(err, axis=1)
    stats = {
        "MAE_z": float(np.mean(np.abs(err[:, 0]))),
        "MAE_y": float(np.mean(np.abs(err[:, 1]))),
        "MAE_x": float(np.mean(np.abs(err[:, 2]))),
        "RMSE":  float(np.sqrt((err**2).mean())),
        "p50":   float(np.percentile(mag, 50)),
        "p95":   float(np.percentile(mag, 95)),
        "max":   float(mag.max()),
    }
    print(f"== {label} ==")
    print(
        f"  POIs: {n_valid_interior} interior-valid / {n_valid_all} valid-all / {n_total} total"
    )
    for status_value in POIStatus:
        count = int(np.count_nonzero(result.status == status_value))
        if count:
            print(f"    {status_value.name:12s} {count}")
    for k, v in stats.items():
        print(f"  {k:6s} {v:.4f} voxels")
    print()

In [ ]:
reference = make_texture(shape, sigma=1.5, seed=0)
print(
    f"reference {reference.shape} {reference.dtype}, "
    f"mean={reference.mean():.3e}, std={reference.std():.3f}"
)

## 1. Integer rigid shift `(2, -3, 1)`

Should recover exactly. Failure here means an integer-peak / sign-convention
bug, not a sub-voxel fit issue.

In [ ]:
field_fn = rigid_shift((2.0, -3.0, 1.0))
pair = make_pair(shape, field_fn, reference=reference)
result = correlate(
    pair.reference, pair.deformed,
    window=window, overlap=overlap, search_radius=window // 2,
)
report("integer rigid (2, -3, 1)", result, field_fn)

## 2. Fractional rigid shift `(0.5, -0.3, 0.7)`

Plan budget assertion: per-axis error `< 0.1` voxel on textured noise.

In [ ]:
field_fn = rigid_shift((0.5, -0.3, 0.7))
pair = make_pair(shape, field_fn, reference=reference)
result = correlate(
    pair.reference, pair.deformed,
    window=window, overlap=overlap, search_radius=window // 2,
)
report("fractional rigid (0.5, -0.3, 0.7)", result, field_fn)

## 3. Showcase rigid shift `(1.5, 0.0, -2.0)`

Same shift as the failing showcase; phantom replaced by the band-limited
texture. If per-axis MAE here is sub-0.1 the residual error in the showcase
is phantom-induced; if it stays high there is a core algorithm issue.

In [ ]:
field_fn = rigid_shift((1.5, 0.0, -2.0))
pair = make_pair(shape, field_fn, reference=reference)
result = correlate(
    pair.reference, pair.deformed,
    window=window, overlap=overlap, search_radius=window // 2,
)
report("showcase rigid (1.5, 0.0, -2.0)", result, field_fn)

## 4. Showcase composite (rigid + sinusoidal)

Reproduces the original showcase warp on the texture-only reference. Any
remaining excess error vs case 3 is attributable to intra-window strain
from the sinusoidal component (wavelength 60 vs window 32).

In [ ]:
field_fn = compose(
    rigid_shift((1.5, 0.0, -2.0)),
    sinusoidal(amplitude=(0.0, 1.5, 1.5), wavelength=(1.0, 60.0, 60.0)),
)
pair = make_pair(shape, field_fn, reference=reference)
result = correlate(
    pair.reference, pair.deformed,
    window=window, overlap=overlap, search_radius=window // 2,
)
report("showcase composite (rigid + sinusoidal)", result, field_fn)

## 6. Hypothesis test — apodization-mismatch bias

Cases 1 + 3 both show a per-axis MAE that scales with the **integer**
part of the shift, while case 2 (sub-voxel-only) is clean. The proposed
mechanism is that the Tukey window is applied at the same start index
in reference and deformed subvolumes, so when the true displacement is
`u != 0` the two windowed crops carry different apodizations of the
underlying texture. The bias should therefore:

- **shrink as `tukey_alpha → 0`** (rectangular window → no mismatch),
- **shrink as `window` grows relative to `|u|`** (overlap fraction
  `(W - |u|) / W` rises, residual mismatch drops).

The two sweeps below probe each effect independently. Truth values are
the showcase `(1.5, 0.0, -2.0)` — covers integer (x = -2), half-voxel
(z = 1.5), and zero (y = 0) on the same run.

In [ ]:
import dataclasses

# Smaller fixed band: only stays away from the warp's reflect-mode boundary
# (max displacement + a few voxels of spline support). Window-driven extraction
# bounds are already enforced upstream by `build_grid`, so we don't need to
# pad by `window_size` here — that's what NaN'd the previous sweep.
sweep_band = 8


@dataclasses.dataclass(frozen=True)
class Stats:
    mae_z: float
    mae_y: float
    mae_x: float
    bias_z: float  # signed mean error (recovered - truth); reveals shrinkage
    bias_y: float
    bias_x: float
    rmse: float
    p95: float
    n_valid: int
    n_outlier: int


def measure(shift, *, window_size, tukey_alpha, deformed=None):
    """Run correlate() and return per-axis MAE + signed bias.

    If `deformed` is supplied, it is used directly (e.g. an `np.roll`'d
    reference). Otherwise the cubic-spline warp is invoked.
    """
    field_fn = rigid_shift(shift)
    pair = make_pair(shape, field_fn, reference=reference)
    ref_vol = pair.reference
    def_vol = deformed if deformed is not None else pair.deformed
    result = correlate(
        ref_vol, def_vol,
        window=window_size, overlap=overlap, search_radius=window_size // 2,
        tukey_alpha=tukey_alpha,
    )
    truth = field_fn(result.positions)
    interior = interior_mask(result.positions, shape, sweep_band)
    valid = result.valid & interior
    err = (result.displacements - truth)[valid]
    if err.size == 0:
        nan = float("nan")
        return Stats(nan, nan, nan, nan, nan, nan, nan, nan, 0, 0)
    mag = np.linalg.norm(err, axis=1)
    return Stats(
        mae_z=float(np.mean(np.abs(err[:, 0]))),
        mae_y=float(np.mean(np.abs(err[:, 1]))),
        mae_x=float(np.mean(np.abs(err[:, 2]))),
        bias_z=float(np.mean(err[:, 0])),
        bias_y=float(np.mean(err[:, 1])),
        bias_x=float(np.mean(err[:, 2])),
        rmse=float(np.sqrt((err ** 2).mean())),
        p95=float(np.percentile(mag, 95)),
        n_valid=int(valid.sum()),
        n_outlier=int((result.status == POIStatus.OUTLIER).sum()),
    )


def print_table(rows, header):
    print(header)
    print(
        f"  {'param':>10}  {'MAE_z':>7}  {'MAE_y':>7}  {'MAE_x':>7}  "
        f"{'bias_z':>7}  {'bias_y':>7}  {'bias_x':>7}  "
        f"{'RMSE':>7}  {'n_ok':>5}"
    )
    for label, stats in rows:
        print(
            f"  {label:>10}  {stats.mae_z:7.4f}  {stats.mae_y:7.4f}  "
            f"{stats.mae_x:7.4f}  {stats.bias_z:+7.4f}  "
            f"{stats.bias_y:+7.4f}  {stats.bias_x:+7.4f}  "
            f"{stats.rmse:7.4f}  {stats.n_valid:5d}"
        )
    print()

### 6a. Tukey-alpha sweep at fixed `window=32`

Hypothesis: MAE on the integer-shifted axis (x) should drop monotonically
as `tukey_alpha → 0`. The fractional axis (z) and the zero axis (y)
should change much less.

In [ ]:
shift = (1.5, 0.0, -2.0)
tukey_rows = [
    (f"alpha={a}", measure(shift, window_size=32, tukey_alpha=a))
    for a in (0.0, 0.05, 0.25)
]
print_table(tukey_rows, f"shift={shift}, window=32, sweep tukey_alpha")

### 6b. Window sweep at fixed `tukey_alpha=0.25`

Hypothesis: MAE on the integer-shifted axis (x) should shrink roughly
as `|u| / W` decreases, so doubling `window` from 32 to 64 should
roughly halve the bias.

In [ ]:
window_rows = [
    (f"window={w}", measure(shift, window_size=w, tukey_alpha=0.25))
    for w in (32, 48, 64)
]
print_table(window_rows, f"shift={shift}, tukey_alpha=0.25, sweep window")

### 6c. Verdict

Plot both sweeps side-by-side and check the predicted scaling. If MAE_x
in 6a falls toward zero at `alpha=0` and MAE_x in 6b roughly halves
between `window=32` and `window=64`, the apodization-mismatch hypothesis
holds and the fix is one of {lower default `tukey_alpha`, larger default
`window`, Padfield masked FFT}. If MAE_x stays high in both sweeps, the
bias has another source (extraction indexing, mean-subtraction, or
normalization).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex="col")

alphas = [0.0, 0.05, 0.25]
windows = [32, 48, 64]

# Top row: MAE
axes[0, 0].plot(alphas, [r.mae_z for _, r in tukey_rows], marker="o", label="z (truth 1.5)")
axes[0, 0].plot(alphas, [r.mae_y for _, r in tukey_rows], marker="o", label="y (truth 0.0)")
axes[0, 0].plot(alphas, [r.mae_x for _, r in tukey_rows], marker="o", label="x (truth -2.0)")
axes[0, 0].set_ylabel("MAE [voxels]")
axes[0, 0].set_title("6a. Tukey-alpha sweep (window=32)")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(windows, [r.mae_z for _, r in window_rows], marker="o", label="z (truth 1.5)")
axes[0, 1].plot(windows, [r.mae_y for _, r in window_rows], marker="o", label="y (truth 0.0)")
axes[0, 1].plot(windows, [r.mae_x for _, r in window_rows], marker="o", label="x (truth -2.0)")
axes[0, 1].set_title("6b. Window sweep (alpha=0.25)")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Bottom row: signed bias (mean signed error). Reveals shrinkage-toward-zero.
axes[1, 0].axhline(0.0, color="k", lw=0.5)
axes[1, 0].plot(alphas, [r.bias_z for _, r in tukey_rows], marker="o", label="z")
axes[1, 0].plot(alphas, [r.bias_y for _, r in tukey_rows], marker="o", label="y")
axes[1, 0].plot(alphas, [r.bias_x for _, r in tukey_rows], marker="o", label="x")
axes[1, 0].set_xlabel("tukey_alpha")
axes[1, 0].set_ylabel("signed bias (recovered - truth) [voxels]")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].axhline(0.0, color="k", lw=0.5)
axes[1, 1].plot(windows, [r.bias_z for _, r in window_rows], marker="o", label="z")
axes[1, 1].plot(windows, [r.bias_y for _, r in window_rows], marker="o", label="y")
axes[1, 1].plot(windows, [r.bias_x for _, r in window_rows], marker="o", label="x")
axes[1, 1].set_xlabel("window size [voxels]")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSigned-bias direction (negative on z, positive on x = shrinkage toward zero):")
for label, r in tukey_rows + window_rows:
    print(f"  {label:>10}  bias_z={r.bias_z:+.4f}  bias_y={r.bias_y:+.4f}  bias_x={r.bias_x:+.4f}")

## 7. Warp-bypass test — is the bias in `warp()` or `correlate()`?

The alpha=0 result rules out apodization mismatch. The remaining suspects
are the **cubic-spline warp** (which prefilters, evaluates the spline,
and applies reflect-mode boundary handling) and the **cyclic-FFT NCC**
itself.

This test substitutes `np.roll(reference, shift_int)` for the warp output
on integer shifts. `np.roll` is an exact, non-interpolating shift with
periodic boundary handling — no spline, no reflect, no resampling error.
If MAE collapses to ≈ 0 here, the bias was warp-induced. If MAE stays
~0.13, the bias is in `correlate()` itself and we need to instrument
the FFT NCC core.

In [ ]:
def roll_shift(ref, shift_int):
    """Apply an integer-voxel shift via np.roll. Pull-back convention:
    out[x] = ref[x - shift], so np.roll uses +shift along each axis.
    """
    z, y, x = (int(round(s)) for s in shift_int)
    return np.roll(ref, shift=(z, y, x), axis=(0, 1, 2))


# 7a — confirm np.roll is consistent with warp() for integer shifts in the
# interior (a sanity check that the two are interchangeable as inputs to
# the correlator, modulo the boundary mode).
shift_int = (2, -3, 1)
field_int = rigid_shift([float(s) for s in shift_int])
pair_int = make_pair(shape, field_int, reference=reference)
def_warp = pair_int.deformed
def_roll = roll_shift(reference, shift_int)

# Compare in the interior (avoid boundary mode discrepancy).
inner = (slice(8, -8),) * 3
diff = np.abs(def_warp[inner] - def_roll[inner])
print(
    f"warp vs np.roll  shift={shift_int}: "
    f"L_inf={diff.max():.3e}  RMS={np.sqrt((diff**2).mean()):.3e}"
)

# 7b — feed correlate() with np.roll-derived deformeds at three shifts.
warp_bypass_rows = []
for shift_int in [(2, -3, 1), (1, 0, -2), (3, 0, -2)]:
    def_roll = roll_shift(reference, shift_int)
    stats = measure(
        tuple(float(s) for s in shift_int),
        window_size=32, tukey_alpha=0.25,
        deformed=def_roll,
    )
    warp_bypass_rows.append((f"roll{shift_int}", stats))

# 7c — same shifts via the spline warp, for direct comparison.
warp_full_rows = []
for shift_int in [(2, -3, 1), (1, 0, -2), (3, 0, -2)]:
    stats = measure(
        tuple(float(s) for s in shift_int),
        window_size=32, tukey_alpha=0.25,
    )
    warp_full_rows.append((f"warp{shift_int}", stats))

print_table(warp_bypass_rows, "warp-bypass (np.roll integer shift)")
print_table(warp_full_rows,   "spline warp (cubic + reflect)")

# Verdict line.
def avg_mae(rows):
    return np.mean([
        max(r.mae_z, r.mae_x)  # ignore the zero-shift y axis
        for _, r in rows
    ])

print(
    f"avg max(MAE_z, MAE_x) — np.roll : {avg_mae(warp_bypass_rows):.4f}\n"
    f"avg max(MAE_z, MAE_x) — warp    : {avg_mae(warp_full_rows):.4f}"
)

## 5. Single-POI introspection (case 3)

Picks a POI in the volume interior and prints the integer peak, the
fractional offset, and the 3-sample neighborhood per axis. Lets you
confirm by eye whether the integer bin is right and whether the
log-parabolic fit ordering is consistent across axes.

In [ ]:
from mamba_dvc.core.extract import extract_subvolumes
from mamba_dvc.core.ncc import correlate as cncc
from mamba_dvc.core.ncc import peak_displacement
from mamba_dvc.core.peakfit import gaussian_subvoxel_fit
from mamba_dvc.core.window import preprocess_subvolumes

field_fn = rigid_shift((1.5, 0.0, -2.0))
pair = make_pair(shape, field_fn, reference=reference)
result = correlate(
    pair.reference, pair.deformed,
    window=window, overlap=overlap, search_radius=window // 2,
)

interior = interior_mask(result.positions, shape, boundary_band)
valid_interior_idx = np.flatnonzero(result.valid & interior)
i = int(valid_interior_idx[len(valid_interior_idx) // 2])

half = np.array([(w - 1) / 2.0 for w in result.window], dtype=np.float32)
start = np.round(result.positions[i] - half).astype(np.int64)[None, :]
ref_sv = extract_subvolumes(pair.reference, start, result.window)
def_sv = extract_subvolumes(pair.deformed, start, result.window)
ref_pp = preprocess_subvolumes(ref_sv, None)
def_pp = preprocess_subvolumes(def_sv, None)
corr = cncc(ref_pp, def_pp, mode="cyclic", normalization="global")
integer, _ = peak_displacement(corr)
fractional = gaussian_subvoxel_fit(corr, integer)

wz, wy, wx = corr.shape[1:]
iz = int(integer[0, 0]) % wz
iy = int(integer[0, 1]) % wy
ix = int(integer[0, 2]) % wx
truth = field_fn(result.positions[i : i + 1])[0]
recovered = result.displacements[i]

print(f"POI {i} center {result.positions[i]} truth {truth}")
print(f"  integer peak    {integer[0].tolist()}")
print(f"  fractional fit  {fractional[0].tolist()}")
print(f"  recovered       {recovered.tolist()}")
print(f"  pipeline output {recovered}  (residual {recovered - truth})")
print(f"  z neighbors: {corr[0, (iz - 1) % wz, iy, ix]:.4f}  {corr[0, iz, iy, ix]:.4f}  {corr[0, (iz + 1) % wz, iy, ix]:.4f}")
print(f"  y neighbors: {corr[0, iz, (iy - 1) % wy, ix]:.4f}  {corr[0, iz, iy, ix]:.4f}  {corr[0, iz, (iy + 1) % wy, ix]:.4f}")
print(f"  x neighbors: {corr[0, iz, iy, (ix - 1) % wx]:.4f}  {corr[0, iz, iy, ix]:.4f}  {corr[0, iz, iy, (ix + 1) % wx]:.4f}")

## 8. A/B test — linear vs cyclic NCC across window sizes

Order-1 fix landed: `mamba_dvc.core.ncc` now exposes a zero-padded
linear kernel paired with the Lewis (1995) overlap-aware denominator,
selectable through the pipeline via `ncc_mode` and `ncc_normalization`.
This sweep checks the predicted bias collapse on the same showcase
shift used in section 6 (`(1.5, 0.0, -2.0)`):

- The cyclic+global combination (legacy default) carries the `|u|/W`
  shrinkage bias documented in `docs/insights/error-minimization.md`.
- The linear+overlap combination (new pipeline default) removes it.
- Both should converge as `W` grows; the gap shrinks but does not
  invert.

For each `(window, mode)` pair we report per-axis MAE, the signed bias
(reveals the toward-zero direction), and RMSE on the interior POIs.
Per the per-mode Tukey resolution baked into the pipeline, each mode
runs at its canonical alpha (0.0 for linear, 0.25 for cyclic) so the
comparison is between the kernels as they would actually ship, not
between two flavors of one window function.


In [ ]:
ab_shift = (1.5, 0.0, -2.0)
ab_modes = [
    ("cyclic-global",  {"ncc_mode": "cyclic", "ncc_normalization": "global"}),
    ("linear-overlap", {"ncc_mode": "linear", "ncc_normalization": "overlap"}),
]
ab_windows = [32, 48, 64]


def measure_mode(shift, *, window_size, mode_kwargs):
    field_fn = rigid_shift(shift)
    pair = make_pair(shape, field_fn, reference=reference)
    result = correlate(
        pair.reference, pair.deformed,
        window=window_size, overlap=overlap, search_radius=window_size // 2,
        **mode_kwargs,  # tukey_alpha=None ⇒ per-mode default
    )
    truth = field_fn(result.positions)
    interior = interior_mask(result.positions, shape, sweep_band)
    valid = result.valid & interior
    err = (result.displacements - truth)[valid]
    if err.size == 0:
        nan = float("nan")
        return Stats(nan, nan, nan, nan, nan, nan, nan, nan, 0, 0)
    mag = np.linalg.norm(err, axis=1)
    return Stats(
        mae_z=float(np.mean(np.abs(err[:, 0]))),
        mae_y=float(np.mean(np.abs(err[:, 1]))),
        mae_x=float(np.mean(np.abs(err[:, 2]))),
        bias_z=float(np.mean(err[:, 0])),
        bias_y=float(np.mean(err[:, 1])),
        bias_x=float(np.mean(err[:, 2])),
        rmse=float(np.sqrt((err ** 2).mean())),
        p95=float(np.percentile(mag, 95)),
        n_valid=int(valid.sum()),
        n_outlier=int((result.status == POIStatus.OUTLIER).sum()),
    )


ab_table = {label: [] for label, _ in ab_modes}
for w in ab_windows:
    for label, kwargs in ab_modes:
        ab_table[label].append(measure_mode(ab_shift, window_size=w, mode_kwargs=kwargs))

for label, rows in ab_table.items():
    print_table(
        list(zip([f"window={w}" for w in ab_windows], rows)),
        f"shift={ab_shift}, mode={label}, sweep window",
    )

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)

for label, rows in ab_table.items():
    axes[0].plot(ab_windows, [r.mae_x for r in rows], marker="o", label=f"{label} (x, truth -2)")
    axes[0].plot(ab_windows, [r.mae_z for r in rows], marker="s", linestyle="--", label=f"{label} (z, truth 1.5)")
axes[0].set_ylabel("MAE [voxels]")
axes[0].set_xlabel("window size [voxels]")
axes[0].set_title("MAE vs window — cyclic vs linear+overlap")
axes[0].axhline(0.1, color="k", lw=0.5, linestyle=":", label="plan budget 0.1 vx")
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].axhline(0.0, color="k", lw=0.5)
for label, rows in ab_table.items():
    axes[1].plot(ab_windows, [r.bias_x for r in rows], marker="o", label=f"{label} (x bias)")
    axes[1].plot(ab_windows, [r.bias_z for r in rows], marker="s", linestyle="--", label=f"{label} (z bias)")
axes[1].set_ylabel("signed bias (recovered - truth) [voxels]")
axes[1].set_xlabel("window size [voxels]")
axes[1].set_title("Signed bias vs window — sign tracks shrinkage")
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()
